# RiskPO gốc + Qwen3-4B LoRA trên 4 A100

Notebook baseline được tạo từ thiết kế của bạn. Giữ Qwen3-4B, GSM8K-only,
thinking=False, BF16 LoRA rank 8/alpha 16, scorer final_numeric_v2, global batch 20,
rollout 5/câu, learning rate 1e-6, độ dài 1024/1024 và validation tắt.

**RiskPO gốc ở đây là thuật toán:** estimator `grpo_bundle_RVaR_quantile_tracking`,
quantile tracking bật và policy loss `vanilla`. Không gọi advantage/loss QUATRO.
Đây vẫn là thí nghiệm LoRA GSM8K-only, không phải full-parameter hay tái lập bài báo.

**Phần cứng: một máy Linux có đúng 4 GPU A100 được expose cho tiến trình.**
Notebook không tự cấp GPU; phiên Colab chỉ có một GPU sẽ dừng tại preflight.
Không ghép bốn tài khoản/runtime Colab. Có thể dùng Jupyter trên máy chủ 4 A100.

| Cấu hình thực thi | Giá trị |
|---|---|
| Model / method | Qwen/Qwen3-4B / riskpo |
| GPUs / nodes | 4 A100 / 1 |
| Actor | FSDP trên 4 rank, sequence parallel=1 |
| Rollout | vLLM V0, TP=1, 4 engine |
| Global batch / PPO minibatch | 20 / 20 câu hỏi, không nhân thêm 4 |
| Response mỗi bước | GSM8K/easymath: 100; DAPO: 200 |
| Response trên mỗi actor rank | GSM8K/easymath: 25; DAPO: 50 |
| Actor microbatch | 1 response/GPU |
| vLLM max_num_seqs | 24 mỗi engine, tối đa 96 toàn máy; còn phụ thuộc token budget |
| Train / test | Split gốc, không subset hoặc chia lại; Telemath ngoài train/tuning |
| Chế độ mặc định | full: 200 bước GSM8K/easymath; 500 DAPO |
| Smoke tùy chọn | 5 bước, cùng batch/rollout/độ dài |
| Checkpoint / session | Save mỗi 5 bước, giữ 2 backup hợp lệ, pause sau 40 bước mới |

`full` vẫn chỉ cập nhật adapter LoRA. Train reward không phải điểm validation.
Tăng số GPU giữ cùng ngân sách dữ liệu nhưng có thể thay đổi thứ tự batching và
kết quả số học; không bảo đảm nhanh gấp 4 hay bit-for-bit giống run một GPU.

## 1. Cấu hình run và kiểm tra bốn GPU

Mặc định giữ đúng model/dataset/LoRA/thinking của notebook nguồn, đổi sang RiskPO
và 4 A100. `METHOD="riskpo"` được khóa để tránh bật nhầm hybrid.
`RUN_PROFILE="full"`; có thể chọn `smoke` để thử 5 bước trước.

`RESUME_BACKUP_ROOT=""` khởi tạo run mới. Chỉ resume checkpoint do notebook 4 GPU
này tạo, cùng world size, model, dữ liệu và cấu hình. Không dùng backup hybrid
hoặc backup một GPU từ notebook cũ.

Đặt `RISKPO_WORK_ROOT` cho ổ làm việc, `RISKPO_OUTPUT_ROOT` cho ổ bền vững và
`RISKPO_CHECKPOINT_ROOT` cho ổ local đủ dung lượng trước khi chạy cell.
Nếu server có hơn 4 GPU, đặt `CUDA_VISIBLE_DEVICES` thành 4 GPU đã được cấp.
Notebook dùng Python riêng; không chạy đồng thời hai job vào cùng backup root.

## Checkpoint qua nhiều phiên trên cùng cấu hình 4 A100

`DURABLE_CHECKPOINTS=True`: lưu mỗi 5 bước, kiểm tra đầy đủ shard của rank 0–3,
fsdp_config world_size=4, SHA-256, rồi công bố `complete.json`. Giữ hai backup đã
xác minh; bản cũ hơn chỉ bị xóa sau khi có bản mới hợp lệ. `.partial-*` không resume.

Lần đầu để RESUME_BACKUP_ROOT rỗng. Phiên sau đặt đúng backup root được in ra,
giữ cùng 4 GPU, model, method, dữ liệu, LoRA, thinking, profile và môi trường.
Checkpoint một GPU/hybrid không tương thích. Nếu không có bản hợp lệ thì dừng;
không âm thầm bắt đầu lại từ base. Checkpoint mới hỏng thì thử bản hợp lệ trước đó.

Full có tổng 200/500 bước. SESSION_STEPS=40 dừng sau một bản sao đã kiểm tra,
không thay tổng bước hoặc lịch learning rate. Có thể resume ở 40 → 80 → … → 200.
Không dùng checkpoint smoke để tiếp tục full vì tổng bước và fingerprint khác.

Lưu cả model/optimizer/scheduler/RNG của bốn worker, dataloader và trạng thái
driver, gồm q/optimizer_q/is_warmup của RiskPO. Adapter dùng để suy luận;
muốn resume đầy đủ phải dùng toàn bộ checkpoint. Không bảo đảm bit-for-bit.

Chọn ổ persistent cho OUTPUT_ROOT, đủ hai backup và bản đang copy. Backup không
bảo đảm storage không mất dữ liệu về sau; nếu runtime mất giữa copy, resume từ
bản hoàn chỉnh gần nhất. Mốc 40 bước không phải giới hạn giờ cứng.

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import datetime
import hashlib
import warnings

REPO_URL = "https://github.com/Belldenchoi/RiskPO.git"
REPO_REF = '68ed7c5898b009fe7e392c2b01feb85b3bb2ff18'
DATASET = "gsm8k"  # Chỉ GSM8K train/test; tùy chọn cũ: "easymath", "dapo"
METHOD = 'riskpo'
LORA_RANK = 8
LORA_ALPHA = 16
TRAIN_BATCH_SIZE = 20
PPO_MINI_BATCH_SIZE = 20
USE_REFERENCE_MODEL = False
MODEL_ID = "Qwen/Qwen3-4B"
ENABLE_THINKING = False  # Qwen3: False/True; model khác: None (template mặc định)
RUN_PROFILE = "full"  # "smoke": 5 bước; "full": 200 gsm8k/easymath / 500 dapo, vẫn LoRA
DURABLE_CHECKPOINTS = True
SESSION_STEPS = 40  # stop after verified backup; total target stays 200/500
RESUME_BACKUP_ROOT = ''
USE_DRIVE = "google.colab" in sys.modules

WORK_ROOT = Path(os.environ.get("RISKPO_WORK_ROOT", str(Path("/content") if Path("/content").is_dir() else Path.cwd() / "riskpo_4a100_runtime"))).expanduser().resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "RiskPO-4A100"
PY = WORK_ROOT / "riskpo-4a100-env/bin/python"
DATA_DIR = WORK_ROOT / "data/riskpo_reference"
ENV = dict(os.environ, VLLM_USE_V1="0", HYDRA_FULL_ERROR="1",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
assert DATASET in {"gsm8k", "easymath", "dapo"}
assert METHOD == "riskpo", "Notebook này dành cho baseline RiskPO gốc"
assert RUN_PROFILE in {"smoke", "full"}
REQUIRED_GPUS = 4
if USE_REFERENCE_MODEL:
    MODEL_ID = ("Qwen/Qwen2.5-1.5B-Instruct" if DATASET != "dapo"
                else "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
assert ENABLE_THINKING is None or type(ENABLE_THINKING) is bool
if ENABLE_THINKING is not None and not MODEL_ID.startswith("Qwen/Qwen3-"):
    raise ValueError("Thinking override này chỉ dành cho Qwen3; model khác đặt ENABLE_THINKING=None.")

def run(args, **kwargs):
    kwargs.setdefault("env", ENV)
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

def run_py(source):
    return run([PY, "-c", source], cwd=REPO)

def check_gpu_count():
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True
    )
    print(gpu_info)
    if len(gpu_info.strip().splitlines()) < REQUIRED_GPUS:
        raise RuntimeError(
            "Cần bốn GPU A100 trên cùng máy Linux. Kiểm tra GPU allocation trước khi cài đặt."
        )
    return gpu_info

# Dừng sớm nếu không có GPU; bước 4 kiểm tra thêm khả năng BF16/FlashAttention.
if not sys.platform.startswith("linux"):
    raise RuntimeError("Notebook training này yêu cầu Linux/CUDA.")
ENV.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3")
GPU_INFO = check_gpu_count()
run(["free", "-h"])
run(["df", "-h", WORK_ROOT])
print("Model:", MODEL_ID, "| Method:", METHOD, "| Dataset:", DATASET,
      "| enable_thinking:", ENABLE_THINKING)

## 2. Clone phiên bản code đã đối chiếu và tạo Python 3.10 riêng

Clone riêng vào `RiskPO-4A100`, cố định commit trong `REPO_REF`. Không ghi đè
repo hay tự đổi nhánh của clone đã tồn tại; kiểm tra HEAD để tránh resume với code khác.
Notebook nguồn và checkout trên máy của bạn được giữ nguyên.

In [ ]:
if not REPO.exists():
    run(["git", "clone", REPO_URL, REPO])
    if REPO_REF != "main":
        run(["git", "checkout", "--detach", REPO_REF], cwd=REPO)
elif not (REPO / ".git").is_dir():
    raise RuntimeError("/content/RiskPO đã tồn tại nhưng không phải Git repo.")
print("Remote:")
run(["git", "remote", "-v"], cwd=REPO)
run(["git", "log", "-1", "--oneline"], cwd=REPO)
core = (REPO / "verl/trainer/ppo/core_algos.py").read_text()
assert "compute_grpo_bundle_rvar_outcome_advantage_quantile_tracking" in core, "Thiếu RiskPO estimator."
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if actual_commit != REPO_REF:
    raise RuntimeError("Clone hiện có không khớp REPO_REF; dùng WORK_ROOT mới hoặc kiểm tra code trước khi tiếp tục.")

run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv")
assert UV, "Không tìm thấy uv sau cài đặt."
run([UV, "python", "install", "3.10"])
if not PY.exists():
    run([UV, "venv", "--python", "3.10", "--seed", PY.parent.parent])
run([PY, "--version"])

## 3. Cài dependencies

Giữ bộ phiên bản tương thích với code repo hiện tại; thêm TensorBoard đúng backend logging của script RiskPO. Không cần cài tensorboard vào kernel Colab vì training chạy bằng Python riêng.

Nguồn phiên bản: [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), [vLLM requirements](https://github.com/vllm-project/vllm/blob/v0.8.5.post1/requirements/cuda.txt), [FlashAttention wheel](https://github.com/Dao-AILab/flash-attention/releases/tag/v2.7.4.post1).


In [ ]:
run([UV, "pip", "install", "--python", PY,
     "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
     "--index-url", "https://download.pytorch.org/whl/cu124"])

pins = [
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "vllm==0.8.5.post1", "transformers==4.51.3",
    "peft==0.15.2", "accelerate==1.6.0", "ray[default]==2.43.0",
    "tensordict==0.8.3", "torchdata==0.11.0",
    "datasets==3.6.0", "numpy==1.26.4", "hydra-core==1.3.2",
]
wheel = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/"
    "v2.7.4.post1/"
    "flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp310-cp310-linux_x86_64.whl"
)
run([UV, "pip", "install", "--python", PY, *pins,
     wheel, "tensorboard", "-e", str(REPO) + "[math]"])
run([UV, "pip", "check", "--python", PY])


## 4. Import, kiểm tra A100 và scorer

Yêu cầu đúng 4 GPU visible, tất cả là A100 với BF16/FlashAttention-2.
Actor dùng BF16 LoRA thông thường, không phải QLoRA. Tiếp theo thử NCCL all-reduce
qua cả 4 GPU trước khi tải model. Các kiểm tra này chưa chứng minh training/save hoạt động.

Giữ nguyên source scorer GSM8K `final_numeric_v2` của notebook nguồn; MATH/DAPO
vẫn route về scorer repo. Không ghi đè scorer trong repo.

In [ ]:
def run_py(source):
    result = subprocess.run(
        [str(PY), "-u", "-c", source],
        cwd=str(REPO),
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Kiểm tra thất bại, exit={result.returncode}. "
            "Xem traceback ngay phía trên."
        )
    return result

In [ ]:
GSM8K_REWARD_SOURCE = r'''# Copyright 2024 Bytedance Ltd. and/or its affiliates
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import re
from decimal import Decimal, InvalidOperation

_SOLUTION_CLIP_CHARS = 300
GSM8K_SCORER_VERSION = "final_numeric_v2"
_ANSWER_MARKER = re.compile(r"####(?!#)|\\boxed\b")
_NUMBER = re.compile(r"[+-]?(?:(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?")


def _numeric_token(value):
    """Accept a scalar number, not units, expressions, or arbitrary answer text."""
    token = str(value).strip()
    if token.startswith("$") and token.endswith("$"):
        token = token[1:-1].strip()
    if len(token) > 256 or _NUMBER.fullmatch(token) is None:
        return None
    return token.replace(",", "")


def _extract_strict(solution_str):
    # Only the final-response portion counts for models with explicit thinking tags.
    # A correct intermediate answer in <think> must not earn reward when the final
    # response is missing, truncated, or wrong.
    final_response = solution_str.rsplit("</think>", 1)[-1]
    if "<think>" in final_response:
        return None

    # Search the whole final response: the answer may be followed by >300 chars.
    # Do not fall back to an earlier correct answer if the last marker is invalid.
    marker = None
    for match in _ANSWER_MARKER.finditer(final_response):
        marker = match
    if marker is None:
        return None
    tail = final_response[marker.end():].lstrip(" \t")
    if marker.group().startswith("####"):
        return _numeric_token(tail.splitlines()[0] if tail else "")

    tail = tail.lstrip()
    if not tail.startswith("{"):
        return None
    depth = 0
    for index, char in enumerate(tail):
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return _numeric_token(tail[1:index])
    return None


def extract_solution(solution_str, method="strict"):
    assert method in ["strict", "flexible"]

    if method == "strict":
        return _extract_strict(solution_str)

    # Preserve the legacy flexible mode; training uses strict explicit markers.
    if len(solution_str) > _SOLUTION_CLIP_CHARS:
        solution_str = solution_str[-_SOLUTION_CLIP_CHARS:]

    answer = re.findall("(\\-?[0-9\\.\\,]+)", solution_str)
    final_answer = None
    for candidate in reversed(answer):
        if candidate not in ["", "."]:
            final_answer = candidate
            break
    return final_answer


def compute_score(solution_str, ground_truth, method="strict", format_score=0.0, score=1.0):
    """The scoring function for GSM8k.

    Reference: Trung, Luong, et al. "Reft: Reasoning with reinforced fine-tuning." Proceedings of the 62nd Annual
    Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 2024.

    Args:
        solution_str: the solution text
        ground_truth: the ground truth
        method: 'strict' accepts a final boxed scalar or #### numeric answer;
            'flexible' retains the legacy last-number heuristic.
        format_score: the score for the format
        score: the score for the correct answer
    """
    answer = extract_solution(solution_str=solution_str, method=method)
    answer = _numeric_token(answer) if answer is not None else None
    expected = _numeric_token(ground_truth)
    if answer is None or expected is None:
        return 0
    try:
        # Exact decimal equality, not an approximate float tolerance or eval().
        # This treats 10, 10.0 and 1e1 as the same numerical answer.
        correct = Decimal(answer) == Decimal(expected)
    except InvalidOperation:
        return 0
    return score if correct else format_score

def compute_dataset_score(data_source, solution_str, ground_truth, extra_info=None, **kwargs):
    """Use the corrected GSM8K scorer; leave MATH/DAPO and other routes unchanged."""
    if data_source == "openai/gsm8k":
        return compute_score(solution_str, ground_truth)
    from verl.utils.reward_score import default_compute_score
    return default_compute_score(
        data_source=data_source, solution_str=solution_str,
        ground_truth=ground_truth, extra_info=extra_info, **kwargs
    )
'''
GSM8K_REWARD_PATH = WORK_ROOT / "riskpo_reward_final_numeric_v2.py"
if GSM8K_REWARD_PATH.exists():
    if GSM8K_REWARD_PATH.read_text(encoding="utf-8") != GSM8K_REWARD_SOURCE:
        raise RuntimeError("Custom reward path đã có nội dung khác; đổi tên file để giữ bản cũ.")
else:
    GSM8K_REWARD_PATH.write_text(GSM8K_REWARD_SOURCE, encoding="utf-8")

run_py("REQUIRED_GPUS = " + repr(REQUIRED_GPUS) + "\n"
       + "REWARD_PATH = " + repr(str(GSM8K_REWARD_PATH)) + "\n" + r'''
import torch
import transformers
import vllm
import flash_attn
import tensorboard
from verl.trainer.ppo.core_algos import (
    compute_grpo_bundle_rvar_outcome_advantage_quantile_tracking, compute_policy_loss,
)
import importlib.util
spec = importlib.util.spec_from_file_location("riskpo_reward_v2_check", REWARD_PATH)
reward = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reward)
default_compute_score = reward.compute_dataset_score

print("torch", torch.__version__, "CUDA", torch.version.cuda)
print("transformers", transformers.__version__, "vllm", vllm.__version__)
print("flash_attn", flash_attn.__version__)
assert torch.cuda.device_count() == REQUIRED_GPUS == 4, "Expose đúng bốn GPU qua CUDA_VISIBLE_DEVICES."
for i in range(REQUIRED_GPUS):
    print(i, torch.cuda.get_device_name(i))
    assert "A100" in torch.cuda.get_device_name(i), "Cấu hình này dành cho bốn A100."
    assert torch.cuda.get_device_capability(i)[0] >= 8, "Cần GPU hỗ trợ FlashAttention-2/BF16."
assert default_compute_score("openai/gsm8k", "#### 42", "42") == 1
assert default_compute_score("openai/gsm8k", r"\boxed{42}", "42") == 1
assert default_compute_score("openai/gsm8k", r"\boxed{41}", "42") == 0
assert default_compute_score("openai/gsm8k", r"<think>\boxed{42}</think>No final answer.", "42") == 0
assert default_compute_score("openai/gsm8k", r"<think>\boxed{41}</think>Final: \boxed{42}", "42") == 1
print("GSM8K scorer:", reward.GSM8K_SCORER_VERSION)
assert default_compute_score(
    "DigitalLearningGmbH/MATH-lighteval", r"\boxed{42}", "42"
) == 1
print("Import/GPU/reward checks passed; chưa kiểm tra đủ VRAM cho training.")
''')

In [ ]:
DISTRIBUTED_PREFLIGHT_SOURCE = 'import os\nfrom datetime import timedelta\nimport torch\nimport torch.distributed as dist\nrank = int(os.environ["LOCAL_RANK"])\ntorch.cuda.set_device(rank)\ndist.init_process_group("nccl", timeout=timedelta(seconds=120))\ntry:\n    assert dist.get_world_size() == 4\n    value = torch.tensor([float(dist.get_rank() + 1)], device=f"cuda:{rank}")\n    dist.all_reduce(value)\n    torch.cuda.synchronize()\n    assert value.item() == 10.0\n    print(f"NCCL_OK rank={dist.get_rank()} GPU={torch.cuda.get_device_name(rank)} sum={value.item()}", flush=True)\nfinally:\n    dist.destroy_process_group()\n'
preflight_path = WORK_ROOT / "riskpo_four_gpu_preflight.py"
preflight_path.write_text(DISTRIBUTED_PREFLIGHT_SOURCE, encoding="utf-8")
run([PY, "-m", "torch.distributed.run", "--standalone", "--nnodes=1",
     "--nproc-per-node=4", preflight_path], cwd=REPO, timeout=180)


## 5. Log và checkpoint trên ổ bền vững

Trên máy chủ: đặt `RISKPO_OUTPUT_ROOT` vào ổ persistent/shared đã được cấp.
Nếu không đặt, mặc định ở WORK_ROOT; tự xác nhận volume này còn tồn tại khi phiên kết thúc.
Drive chỉ được mount nếu chạy trong Colab có đủ GPU.

Checkpoint gồm 12 file shard của 4 rank (model/optimizer/extra), dataloader,
trạng thái driver/quantile và adapter. Nó có thể lớn hơn nhiều so với adapter.
Local và backup giữ hai checkpoint; cần thêm dung lượng cho bản đang sao lưu.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/RiskPO-runs")
else:
    OUTPUT_ROOT = Path(os.environ.get("RISKPO_OUTPUT_ROOT", str(WORK_ROOT / "RiskPO-runs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = Path(os.environ.get("RISKPO_CHECKPOINT_ROOT", str(WORK_ROOT / "checkpoints"))).expanduser().resolve()  # Đổi sang ổ bền vững đủ dung lượng nếu cần.
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        ENV["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN.")
except Exception:
    pass
print("Logs:", OUTPUT_ROOT)
print("Local checkpoints (có thể lớn dù dùng LoRA):", CHECKPOINT_ROOT)

## 6. Chuẩn bị dữ liệu — mặc định chỉ GSM8K train/test

Nhánh gsm8k dùng bộ chuẩn bị tự chứa trong notebook, chỉ tải `openai/gsm8k`, config `main`. Giữ cách tạo prompt boxed, ground truth và extra_info như nhánh GSM8K của script gốc; không cần push GitHub. Parquet đã tồn tại được giữ lại và kiểm tra ở mục 8. Không lấy subset, chia lại hoặc thêm `/no_think`.

- **gsm8k (mặc định):** train = GSM8K train; val_files = GSM8K test. Không dùng MATH, kể cả khi file MATH còn từ run cũ.
- easymath (tùy chọn): train = GSM8K train + MATH train; val_files = GSM8K test + MATH test.
- dapo: train = DAPO train; val_files = AIME-2024 (dataset upstream đặt split tên train, nhưng vai trò ở RiskPO là evaluation).

Trainer vẫn lọc prompt quá dài theo giới hạn 1024 token như script gốc, nên số mẫu thực dùng có thể ít hơn số hàng parquet. Việc dùng test để theo dõi training theo đúng repo không biến chúng thành holdout độc lập để chọn checkpoint rồi báo điểm không thiên lệch. Giữ TeleMath ngoài tuning.


In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = DATA_DIR / "raw"
GSM8K_PREPROCESS_SOURCE = r'''
from pathlib import Path
import re
import datasets

def make_gsm8k_row(example, idx, split):
    question_raw, answer_raw = example['question'], example['answer']
    match = re.search(r'#### (\-?[0-9\.\,]+)', answer_raw)
    assert match is not None, 'GSM8K answer missing #### numeric ground truth'
    solution = match.group(1).replace(',', '')
    instruction = r"Let's think step by step and output the final answer within \boxed{}."
    return {
        'data_source': 'openai/gsm8k',
        'prompt': [{'role': 'user', 'content': question_raw + ' ' + instruction}],
        'ability': 'math',
        'reward_model': {'style': 'rule', 'ground_truth': solution},
        'extra_info': {'split': split, 'index': idx, 'answer': answer_raw, 'question': question_raw},
    }

def prepare_gsm8k(output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    missing = [split for split in ('train', 'test') if not (output_dir / (split + '.parquet')).exists()]
    if not missing:
        print('Reuse existing GSM8K train/test; contents checked in step 8.')
        return
    dataset = datasets.load_dataset('openai/gsm8k', 'main')
    for split in missing:
        processed = dataset[split].map(make_gsm8k_row, with_indices=True, fn_kwargs={'split': split})
        processed.to_parquet(str(output_dir / (split + '.parquet')))
        print('GSM8K', split, 'rows:', len(processed))
'''
if DATASET == "gsm8k":
    run_py(GSM8K_PREPROCESS_SOURCE + f"\nprepare_gsm8k({str(RAW_DIR / 'gsm8k')!r})\n")
elif DATASET == "easymath":
    for name in ("math", "gsm8k"):
        (RAW_DIR / name).mkdir(parents=True, exist_ok=True)
    run([PY, "data_processing/download_easymath.py",
         "--math_local_dir", RAW_DIR / "math",
         "--gsm8k_local_dir", RAW_DIR / "gsm8k"], cwd=REPO)
else:
    run([PY, "data_processing/download_dapomath.py",
         "--output_dir", RAW_DIR], cwd=REPO)


## 7. Cấu hình RiskPO + LoRA trên 4 GPU

Giữ global batch/PPO minibatch 20 câu hỏi, rollout 5/10, prompt/response 1024/1024
hoặc 1024/3072, learning rate 1e-6, rank 8/alpha 16, offload và validation tắt.
Baseline dùng `grpo_bundle_RVaR_quantile_tracking` + `vanilla`, tracking bật.

Actor FSDP world_size=4, SP=1; rollout TP=1 tạo 4 engine vLLM.
Với GSM8K, global 100 response chia 25/rank; actor microbatch 1.
`max_num_seqs=24` mỗi engine, token budget 2048/4096 và memory utilization 0.6.
Không tăng global batch lên 80. Không dùng QUATRO geometric/trajectory loss.

Giữ `full` 200/500 bước, `smoke` 5 bước. Durable ghi đè lịch lưu thành mỗi 5 bước,
retention=2 và giới hạn phiên 40 bước mới; session limit không sửa tổng bước/LR schedule.

In [ ]:
def build_reference_config(dataset, method, model_id, data_root, checkpoint_dir, run_name):
    """Match executable overrides in the RiskPO scripts, then apply the chosen method."""
    if dataset not in {"gsm8k", "easymath", "dapo"} or method != "riskpo":
        raise ValueError("Unknown dataset or method")
    hard = dataset == "dapo"
    raw = Path(data_root) / "raw"
    train_files = ([str(raw / "dapo_aime2024/dapo-math-17k.parquet")] if hard else
                   [str(raw / "gsm8k/train.parquet"), str(raw / "math/train.parquet")])
    val_files = ([str(raw / "dapo_aime2024/aime-2024.parquet")] if hard else
                 [str(raw / "gsm8k/test.parquet"), str(raw / "math/test.parquet")])
    if dataset == "gsm8k":
        train_files = [str(raw / "gsm8k/train.parquet")]
        val_files = [str(raw / "gsm8k/test.parquet")]
    algorithm = {
        "adv_estimator": "grpo_bundle_RVaR_quantile_tracking",
        "quantile_down": 0.2, "quantile_up": 0.8 if hard else 0.9,
        "bundle_size": 5, "lr_q": 0.1,
        "credit_assign_mode": "sum-mean" if hard else "std",
        "use_q_track_mode": "track", "norm_adv_by_std_in_grpo": True,
        "use_mixing_risk_measure": True, "w_mix": 1.5,
        "use_mean_as_baseline": False, "natural_baseline_base": True,
        "natural_baseline_adv": not hard,
        "quantile_tracking": True, "use_kl_in_reward": False,
    }
    actor = {
        "optim": {"lr": 1e-6},
        "ppo_mini_batch_size": 128 if hard else 512,
        "ppo_micro_batch_size_per_gpu": 64,
        "use_kl_loss": False, "kl_loss_coef": 0.001,
        "kl_loss_type": "low_var_kl", "entropy_coeff": 0,
        "fsdp_config": {"param_offload": hard, "optimizer_offload": False},
        "policy_loss": {"loss_mode": "vanilla"},
        "loss_agg_mode": "token-mean",
    }
    rollout = {
        "name": "vllm", "mode": "sync",
        "log_prob_micro_batch_size_per_gpu": 16 if hard else 64,
        "tensor_model_parallel_size": 2, "gpu_memory_utilization": 0.8,
        "n": 10 if hard else 5,
        "temperature": 1.0, "top_p": 1.0, "top_k": -1,
        "val_kwargs": {"do_sample": False, "n": 1,
                       "temperature": 0.0, "top_p": 1.0, "top_k": -1},
    }
    ref = {
        "log_prob_micro_batch_size_per_gpu": 16 if hard else 64,
        "fsdp_config": {"param_offload": True},
    }
    if hard:
        actor.update(use_dynamic_bsz=False, ppo_max_token_len_per_gpu=4096,
                     ulysses_sequence_parallel_size=4)
        ref.update(log_prob_use_dynamic_bsz=False, log_prob_max_token_len_per_gpu=4096,
                   ulysses_sequence_parallel_size=4)
        rollout.update(log_prob_use_dynamic_bsz=False,
                       log_prob_max_token_len_per_gpu=4096, max_num_batched_tokens=4096)
    return {
        "defaults": ["ppo_trainer", "_self_"],
        "algorithm": algorithm,
        "data": {
            "train_files": train_files, "val_files": val_files,
            "train_batch_size": 512 if hard else 1024,
            "max_prompt_length": 1024, "max_response_length": 3072 if hard else 1024,
            "filter_overlong_prompts": True, "truncation": "error",
        },
        "actor_rollout_ref": {
            "model": {"path": model_id, "lora_rank": 0,
                      "use_remove_padding": True, "enable_gradient_checkpointing": True},
            "actor": actor, "rollout": rollout, "ref": ref,
        },
        "trainer": {
            "critic_warmup": 0, "logger": ["console", "tensorboard"],
            "project_name": "riskpo_reference_" + dataset, "experiment_name": run_name,
            "n_gpus_per_node": 8 if hard else 4, "nnodes": 1,
            "save_freq": 10, "test_freq": 5,
            "total_training_steps": 500 if hard else 200,
            "total_epochs": 30 if hard else 15, "val_before_train": True,
            "default_local_dir": str(checkpoint_dir),
        },
    }

def build_lora_config(dataset, method, model_id, data_root, checkpoint_dir, run_name,
                      lora_rank=8, lora_alpha=16, train_batch_size=20, ppo_mini_batch_size=20,
                      run_profile="full"):
    """Keep original data splits; adapt training to four-GPU BF16 LoRA."""
    if run_profile not in {"smoke", "full"}:
        raise ValueError("run_profile must be smoke or full")
    if lora_rank not in {8, 16, 32, 64} or lora_alpha <= 0:
        raise ValueError("Use a supported LoRA rank (8/16/32/64) and positive alpha")
    if (train_batch_size <= 0 or ppo_mini_batch_size <= 0
            or train_batch_size % ppo_mini_batch_size != 0
            or train_batch_size % 5 != 0):
        raise ValueError("Batch must contain full bundles of 5 and be divisible by PPO mini-batch")
    cfg = build_reference_config(dataset, method, model_id, data_root, checkpoint_dir, run_name)
    cfg["data"].update(train_batch_size=train_batch_size, val_batch_size=4,
                       dataloader_num_workers=2)
    model = cfg["actor_rollout_ref"]["model"]
    model.update(lora_rank=lora_rank, lora_alpha=lora_alpha,
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                 "gate_proj", "up_proj", "down_proj"])
    actor = cfg["actor_rollout_ref"]["actor"]
    actor.update(ppo_mini_batch_size=ppo_mini_batch_size,
                 ppo_micro_batch_size_per_gpu=1, use_dynamic_bsz=False,
                 use_torch_compile=False, ulysses_sequence_parallel_size=1)
    actor["fsdp_config"].update(model_dtype="bf16", param_offload=True, optimizer_offload=True, fsdp_size=4)
    rollout = cfg["actor_rollout_ref"]["rollout"]
    rollout.update(
        tensor_model_parallel_size=1, dtype="bfloat16",
        log_prob_micro_batch_size_per_gpu=1, log_prob_use_dynamic_bsz=False,
        load_format="safetensors", layered_summon=True,
        gpu_memory_utilization=0.6, enforce_eager=True, free_cache_engine=True,
        max_num_seqs=24,
        max_num_batched_tokens=max(1024, cfg["data"]["max_prompt_length"] + cfg["data"]["max_response_length"]),
        engine_kwargs={"vllm": {"max_num_seqs": 24, "swap_space": 2}},
    )
    cfg["actor_rollout_ref"]["ref"].update(
        log_prob_micro_batch_size_per_gpu=1, log_prob_use_dynamic_bsz=False,
        ulysses_sequence_parallel_size=1)
    cfg["trainer"].update(n_gpus_per_node=4, nnodes=1, max_actor_ckpt_to_keep=1,
                          resume_mode="disable", val_before_train=False, test_freq=-1)
    if run_profile == "smoke":
        cfg["trainer"].update(total_training_steps=5, save_freq=5)
    n = cfg["actor_rollout_ref"]["rollout"]["n"]
    if (train_batch_size * n) % 4 or (ppo_mini_batch_size * n) % 4:
        raise ValueError("Global response batch and PPO minibatch must be divisible by four ranks")
    return cfg

In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
THINKING_LABEL = "defaultthink" if ENABLE_THINKING is None else ("think" if ENABLE_THINKING else "nothink")
RUN_NAME = f"{RUN_PROFILE}_{METHOD}_{MODEL_ID.rsplit('/', 1)[-1]}_{THINKING_LABEL}_{DATASET}_{stamp}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
CKPT_DIR = CHECKPOINT_ROOT / RUN_NAME
CONFIG = build_lora_config(
    DATASET, METHOD, MODEL_ID, DATA_DIR, CKPT_DIR, RUN_NAME,
    lora_rank=LORA_RANK, lora_alpha=LORA_ALPHA,
    train_batch_size=TRAIN_BATCH_SIZE, ppo_mini_batch_size=PPO_MINI_BATCH_SIZE,
    run_profile=RUN_PROFILE,
)
# Self-contained custom dataset: works even when the GitHub clone is older.
# Bind the official template variable before RLHFDataset filters/tokenizes.
THINKING_DATASET_SOURCE = r'''
from copy import deepcopy
from verl.utils.dataset.rl_dataset import RLHFDataset

def configured_tokenizer(tokenizer, enabled):
    if type(enabled) is not bool:
        raise ValueError('enable_thinking must be bool')
    template = tokenizer.get_chat_template()
    if 'enable_thinking' not in template:
        raise ValueError('Tokenizer template does not support enable_thinking')
    probe = [{'role': 'user', 'content': 'What is 1 + 1?'}]
    expected = tokenizer.apply_chat_template(
        probe, tokenize=False, add_generation_prompt=True, enable_thinking=enabled)
    configured = deepcopy(tokenizer)
    configured.chat_template = ('{% set enable_thinking = ' +
                                ('true' if enabled else 'false') + ' %}' + template)
    actual = configured.apply_chat_template(probe, tokenize=False, add_generation_prompt=True)
    assert actual == expected, 'Bound template differs from explicit enable_thinking'
    if not enabled:
        assert actual.endswith('<think>\n\n</think>\n\n'), 'Missing Qwen3 empty thinking prefix'
    print('Qwen3 enable_thinking:', enabled, '| prompt suffix:', repr(actual[-100:]))
    return configured

class ThinkingModeDataset(RLHFDataset):
    def __init__(self, data_files, tokenizer, config, processor=None):
        if processor is not None:
            raise ValueError('This notebook thinking override supports text-only Qwen3')
        tokenizer = configured_tokenizer(tokenizer, config.get('enable_thinking'))
        super().__init__(data_files=data_files, tokenizer=tokenizer, config=config, processor=None)
'''
if ENABLE_THINKING is not None:
    THINKING_DATASET_PATH = RUN_DIR / "riskpo_thinking_dataset_v1.py"
    THINKING_DATASET_PATH.write_text(THINKING_DATASET_SOURCE, encoding="utf-8")
    CONFIG["data"]["enable_thinking"] = ENABLE_THINKING
    CONFIG["data"]["custom_cls"] = {
        "path": str(THINKING_DATASET_PATH), "name": "ThinkingModeDataset",
    }
CONFIG["custom_reward_function"] = {
    "path": str(GSM8K_REWARD_PATH),
    "name": "compute_dataset_score",
}
DURABLE_SOURCE = '"""Opt-in four-GPU notebook checkpoint backup and session boundaries.\n\nOnly checkpoints with a complete, verified manifest are resumable. Interrupted\ncopies remain in .partial-* folders and are never selected or pruned. This is\nnot a guarantee against a remote filesystem losing acknowledged writes.\n"""\n\nimport copy\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport time\nimport uuid\nimport warnings\n\nVERSION = 2\nWORLD_SIZE = 4\nMARKER = "complete.json"\nSTEP_PATTERN = re.compile(r"global_step_(\\d+)")\nREQUIRED = (\n    "data.pt", "trainer_state.pt", "actor/fsdp_config.json",\n    "actor/lora_adapter/adapter_config.json", "actor/lora_adapter/adapter_model.safetensors",\n) + tuple(\n    f"actor/{kind}_world_size_{WORLD_SIZE}_rank_{rank}.pt"\n    for rank in range(WORLD_SIZE) for kind in ("model", "optim", "extra_state")\n)\n\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef inventory(folder):\n    folder = Path(folder)\n    result = {}\n    for path in sorted(folder.rglob("*")):\n        if path.is_symlink():\n            raise ValueError(f"Symlink is not allowed in checkpoint: {path}")\n        if path.is_file() and path.name != MARKER:\n            if path.stat().st_size == 0:\n                raise ValueError(f"Empty checkpoint file: {path}")\n            result[path.relative_to(folder).as_posix()] = {\n                "size": path.stat().st_size, "sha256": sha256(path)}\n    if not set(REQUIRED).issubset(result):\n        raise ValueError(f"Incomplete checkpoint: {folder}; missing {set(REQUIRED) - result.keys()}")\n    fsdp = json.loads((folder / "actor/fsdp_config.json").read_text(encoding="utf-8"))\n    if fsdp.get("world_size") != WORLD_SIZE:\n        raise ValueError("Checkpoint world_size must match the four-GPU run")\n    shard_pattern = re.compile(r"actor/(model|optim|extra_state)_world_size_\\d+_rank_\\d+\\.pt")\n    shards = {name for name in result if shard_pattern.fullmatch(name)}\n    expected = {name for name in REQUIRED if shard_pattern.fullmatch(name)}\n    if shards != expected:\n        raise ValueError("Unexpected or missing model/optimizer/RNG shards")\n    return result\n\n\ndef verify(folder, fingerprint):\n    folder = Path(folder)\n    if folder.is_symlink() or not STEP_PATTERN.fullmatch(folder.name):\n        raise ValueError(f"Invalid checkpoint folder: {folder}")\n    manifest = json.loads((folder / MARKER).read_text(encoding="utf-8"))\n    if manifest.get("version") != VERSION or manifest.get("fingerprint") != fingerprint or manifest.get("world_size") != WORLD_SIZE:\n        raise ValueError("Checkpoint version/config/data/code fingerprint mismatch")\n    if manifest.get("step") != int(STEP_PATTERN.fullmatch(folder.name).group(1)):\n        raise ValueError("Checkpoint step mismatch")\n    if inventory(folder) != manifest["files"]:\n        raise ValueError(f"Checkpoint checksum mismatch: {folder}")\n    return manifest\n\n\ndef latest_verified(root, fingerprint):\n    root = Path(root)\n    candidates = sorted(\n        (p for p in root.glob("global_step_*") if STEP_PATTERN.fullmatch(p.name)),\n        key=lambda p: int(STEP_PATTERN.fullmatch(p.name).group(1)), reverse=True)\n    for path in candidates:\n        try:\n            verify(path, fingerprint)\n            return path\n        except (ValueError, OSError, KeyError) as exc:\n            warnings.warn(f"Ignoring invalid backup {path}: {exc}")\n    return None\n\n\ndef publish_checkpoint(source, root, fingerprint, keep=2):\n    """Copy synchronously, verify bytes, publish completion, then prune old backups."""\n    if keep < 2:\n        raise ValueError("Keep at least two verified backups")\n    source, root = Path(source).resolve(), Path(root).resolve()\n    if not STEP_PATTERN.fullmatch(source.name) or source == root or root.is_relative_to(source):\n        raise ValueError("Invalid source or overlapping backup root")\n    root.mkdir(parents=True, exist_ok=True)\n    files = inventory(source)\n    target = root / source.name\n    if target.exists():\n        existing = verify(target, fingerprint)\n        if existing["files"] != files:\n            raise FileExistsError(f"Refusing to overwrite a different checkpoint: {target}")\n        return target\n    staging = root / (".partial-" + source.name + "-" + uuid.uuid4().hex)\n    shutil.copytree(source, staging, ignore=shutil.ignore_patterns(MARKER))\n    if inventory(staging) != files:\n        raise ValueError(f"Backup verification failed; incomplete copy retained at {staging}")\n    manifest = {"version": VERSION, "step": int(STEP_PATTERN.fullmatch(source.name).group(1)),\n                "fingerprint": fingerprint, "files": files, "world_size": WORLD_SIZE}\n    with (staging / MARKER).open("x", encoding="utf-8") as stream:\n        json.dump(manifest, stream, indent=2)\n        stream.flush()\n        os.fsync(stream.fileno())\n    staging.rename(target)\n    verify(target, fingerprint)\n    print(f"DURABLE_CHECKPOINT_READY step={manifest[\'step\']} path={target}", flush=True)\n    # Never delete unknown, corrupt or incomplete folders. Only owned, verified\n    # direct children with the same run fingerprint are eligible for retention.\n    valid = []\n    for path in root.glob("global_step_*"):\n        try:\n            verify(path, fingerprint)\n            valid.append(path)\n        except (ValueError, OSError, KeyError):\n            continue\n    valid.sort(key=lambda p: int(STEP_PATTERN.fullmatch(p.name).group(1)), reverse=True)\n    for path in valid[keep:]:\n        if path.is_symlink() or path.resolve().parent != root or not STEP_PATTERN.fullmatch(path.name):\n            raise ValueError("Unsafe retention target")\n        shutil.rmtree(path)\n        print(f"Pruned old backup {path}; {keep} newer verified backups retained", flush=True)\n    return target\n\n\ndef restore_checkpoint(root, local_root, fingerprint):\n    source = latest_verified(root, fingerprint)\n    if source is None:\n        raise FileNotFoundError("No verified durable checkpoint; refusing to restart from base")\n    local_root = Path(local_root).resolve()\n    local_root.mkdir(parents=True, exist_ok=True)\n    target = local_root / source.name\n    if target.exists():\n        verify(target, fingerprint)\n        return target\n    staging = local_root / (".partial-restore-" + uuid.uuid4().hex)\n    shutil.copytree(source, staging)\n    # The directory name is part of verification, so check inventory before rename.\n    manifest = json.loads((staging / MARKER).read_text(encoding="utf-8"))\n    if inventory(staging) != manifest["files"]:\n        raise ValueError("Restored bytes do not match backup")\n    staging.rename(target)\n    verify(target, fingerprint)\n    return target\n\n\ndef prepare_session(config, backup_root, resume, session_steps, extra_identity):\n    """Validate a stable run contract and select a verified local resume path."""\n    if session_steps <= 0 or session_steps % 5:\n        raise ValueError("session_steps must be a positive multiple of 5")\n    trainer = config["trainer"]\n    if trainer["n_gpus_per_node"] != WORLD_SIZE or trainer["nnodes"] != 1:\n        raise ValueError("This checkpoint protocol requires four GPUs on one node")\n    if config["actor_rollout_ref"]["model"]["lora_rank"] <= 0:\n        raise ValueError("This checkpoint protocol requires LoRA")\n    trainer.update(save_freq=5, max_actor_ckpt_to_keep=2, del_local_ckpt_after_load=False)\n    config["actor_rollout_ref"]["actor"]["checkpoint"] = {\n        "save_contents": ["model", "optimizer", "extra"],\n        "load_contents": ["model", "optimizer", "extra"],\n    }\n    canonical = copy.deepcopy(config)\n    for field in ("experiment_name", "default_local_dir", "resume_mode", "resume_from_path", "durable_checkpoint"):\n        canonical["trainer"].pop(field, None)\n    files = {}\n    for role in ("train_files", "val_files"):\n        files[role] = [sha256(path) for path in canonical["data"].pop(role)]\n    for mapping, key in ((canonical["data"].get("custom_cls", {}), "path"),\n                         (canonical.get("custom_reward_function", {}), "path")):\n        if mapping.get(key):\n            mapping[key] = sha256(mapping[key])\n    payload = {"config": canonical, "data_sha256": files, "extra": extra_identity, "version": VERSION}\n    fingerprint = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n    backup_root = Path(backup_root).resolve()\n    contract = backup_root / "run_contract.json"\n    if resume:\n        old = json.loads(contract.read_text(encoding="utf-8"))\n        if old["fingerprint"] != fingerprint:\n            raise ValueError("Resume refused: model/config/data/code/environment differ from the saved run")\n    else:\n        if backup_root.exists() and any(backup_root.iterdir()):\n            raise FileExistsError("New run requires an empty backup root")\n        backup_root.mkdir(parents=True, exist_ok=True)\n        with contract.open("x", encoding="utf-8") as stream:\n            json.dump({"fingerprint": fingerprint, "identity": payload}, stream, indent=2)\n    start = 0\n    if resume:\n        checkpoint = restore_checkpoint(backup_root, trainer["default_local_dir"], fingerprint)\n        start = int(STEP_PATTERN.fullmatch(checkpoint.name).group(1))\n        if start >= trainer["total_training_steps"]:\n            raise ValueError("This run already reached its total training steps; do not train it again")\n        trainer.update(resume_mode="resume_path", resume_from_path=str(checkpoint))\n    else:\n        trainer.update(resume_mode="disable", resume_from_path=None)\n    trainer["durable_checkpoint"] = {"backup_root": str(backup_root), "fingerprint": fingerprint,\n                                     "session_steps": session_steps, "keep": 2}\n    print(f"SESSION_PLAN start={start} stop={min(start + session_steps, trainer[\'total_training_steps\'])} "\n          f"total={trainer[\'total_training_steps\']} save_every=5 backup_root={backup_root}", flush=True)\n    return start\n\n\ndef install_trainer_hooks(trainer_cls):\n    """Installed inside the Ray TaskRunner, not just the notebook/driver process."""\n    if getattr(trainer_cls, "_durable_hooks_installed", False):\n        return\n    original_save, original_load, original_fit = trainer_cls._save_checkpoint, trainer_cls._load_checkpoint, trainer_cls.fit\n\n    class SessionSaved(Exception):\n        pass\n\n    def load(self):\n        result = original_load(self)\n        options = self.config.trainer.get("durable_checkpoint")\n        if options:\n            self._durable_session_start = self.global_steps\n            if self.global_steps:\n                import random\n                import numpy as np\n                import torch\n                state = torch.load(Path(self.config.trainer.resume_from_path) / "trainer_state.pt",\n                                   map_location="cpu", weights_only=False)\n                if state["step"] != self.global_steps:\n                    raise ValueError("Trainer state step mismatch")\n                random.setstate(state["python_rng"])\n                np.random.set_state(state["numpy_rng"])\n                torch.set_rng_state(state["torch_rng"])\n                if self.quantile_tracking:\n                    self.q.data.copy_(state["q"].to(self.q.device))\n                    self.optimizer_q.load_state_dict(state["optimizer_q"])\n                    self.is_warmup = state["is_warmup"]\n                if hasattr(self, "kl_ctrl_in_reward") and "kl_controller" in state:\n                    self.kl_ctrl_in_reward.__dict__.update(state["kl_controller"])\n        return result\n\n    def save(self):\n        original_save(self)\n        options = self.config.trainer.get("durable_checkpoint")\n        if not options:\n            return\n        import random\n        import numpy as np\n        import torch\n        folder = Path(self.config.trainer.default_local_dir) / f"global_step_{self.global_steps}"\n        state = {"step": self.global_steps, "python_rng": random.getstate(),\n                 "numpy_rng": np.random.get_state(), "torch_rng": torch.get_rng_state()}\n        if self.quantile_tracking:\n            state.update(q=self.q.detach().cpu(), optimizer_q=self.optimizer_q.state_dict(), is_warmup=self.is_warmup)\n        if hasattr(self, "kl_ctrl_in_reward"):\n            state["kl_controller"] = self.kl_ctrl_in_reward.__dict__.copy()\n        torch.save(state, folder / "trainer_state.pt")\n        started = time.monotonic()\n        publish_checkpoint(folder, options["backup_root"], options["fingerprint"], options["keep"])\n        print(f"Durable backup verified in {time.monotonic() - started:.1f}s", flush=True)\n        if (self.global_steps < self.total_training_steps and\n                self.global_steps - self._durable_session_start >= options["session_steps"]):\n            raise SessionSaved()\n\n    def fit(self):\n        try:\n            return original_fit(self)\n        except SessionSaved:\n            print(f"SESSION_PAUSED_AFTER_VERIFIED_CHECKPOINT step={self.global_steps} "\n                  f"total={self.total_training_steps}. Resume the same durable run in the next session.", flush=True)\n\n    trainer_cls._save_checkpoint, trainer_cls._load_checkpoint, trainer_cls.fit = save, load, fit\n    trainer_cls._durable_hooks_installed = True\n'
# Self-contained helper: no GitHub push required.
if RESUME_BACKUP_ROOT and not DURABLE_CHECKPOINTS:
    raise ValueError("Resume requires DURABLE_CHECKPOINTS=True")
DURABLE_BACKUP_ROOT = Path(RESUME_BACKUP_ROOT).expanduser() if RESUME_BACKUP_ROOT else RUN_DIR / "durable_checkpoints"
RESUME_START_STEP = 0
if DURABLE_CHECKPOINTS:
    if USE_DRIVE and not str(DURABLE_BACKUP_ROOT.resolve()).startswith("/content/drive/MyDrive/"):
        raise ValueError("For Colab Drive runs, backups must be inside mounted MyDrive")
    if not USE_DRIVE:
        warnings.warn("No Google Drive: ensure DURABLE_BACKUP_ROOT is on persistent storage")
    helper_path = REPO / "verl/utils/checkpoint/durable_riskpo_4gpu.py"
    if helper_path.exists() and helper_path.read_text(encoding="utf-8") != DURABLE_SOURCE:
        raise RuntimeError("Different durable.py exists; preserve it and review or use a fresh clone")
    if not helper_path.exists():
        helper_path.write_text(DURABLE_SOURCE, encoding="utf-8")
    entry = REPO / "verl/trainer/main_ppo.py"
    entry_source = entry.read_text(encoding="utf-8")
    hook = """        # Opt-in durable four-A100 sessions; installed inside the Ray actor process.
        if config.trainer.get("durable_checkpoint"):
            from verl.utils.checkpoint.durable_riskpo_4gpu import install_trainer_hooks

            install_trainer_hooks(RayPPOTrainer)
"""
    if hook not in entry_source:
        if "install_trainer_hooks" in entry_source:
            raise RuntimeError("Unknown existing durable hook; refusing to overwrite")
        anchor = "        OmegaConf.resolve(config)\n"
        if entry_source.count(anchor) != 1:
            raise RuntimeError("Unknown main_ppo.py layout; cannot safely install durable hook")
        entry.write_text(entry_source.replace(anchor, anchor + "\n" + hook), encoding="utf-8")
    import importlib.util
    spec = importlib.util.spec_from_file_location("riskpo_durable_four_gpu_notebook", helper_path)
    durable = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(durable)
    versions = subprocess.check_output([str(UV), "pip", "freeze", "--python", str(PY)], text=True)
    identity = {
        "model_id": MODEL_ID, "dataset": DATASET, "thinking": ENABLE_THINKING,
        "helper_sha256": hashlib.sha256(DURABLE_SOURCE.encode()).hexdigest(),
        "dependencies": sorted(line for line in versions.splitlines() if not line.startswith("-e ")),
        "training_code": {name: hashlib.sha256((REPO / name).read_bytes()).hexdigest() for name in (
            "verl/trainer/main_ppo.py", "verl/trainer/ppo/ray_trainer.py", "verl/trainer/ppo/core_algos.py",
            "verl/workers/fsdp_workers.py", "verl/utils/checkpoint/fsdp_checkpoint_manager.py")},
    }
    RESUME_START_STEP = durable.prepare_session(
        CONFIG, DURABLE_BACKUP_ROOT, bool(RESUME_BACKUP_ROOT), SESSION_STEPS, identity)
    (RUN_DIR / "durable.py").write_text(DURABLE_SOURCE, encoding="utf-8")
    (RUN_DIR / "resume_info.json").write_text(json.dumps({
        "backup_root": str(DURABLE_BACKUP_ROOT), "start_step": RESUME_START_STEP,
        "session_stop": min(RESUME_START_STEP + SESSION_STEPS, CONFIG["trainer"]["total_training_steps"]),
    }, indent=2), encoding="utf-8")
    print("NEXT SESSION: RESUME_BACKUP_ROOT =", repr(str(DURABLE_BACKUP_ROOT)))
CONFIG_NAME = "riskpo_original_lora_4a100"
CONFIG_PATH = REPO / "verl/trainer/config" / (CONFIG_NAME + ".yaml")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
shutil.copy2(CONFIG_PATH, RUN_DIR / CONFIG_PATH.name)
shutil.copy2(GSM8K_REWARD_PATH, RUN_DIR / GSM8K_REWARD_PATH.name)
ENV["TENSORBOARD_DIR"] = str(RUN_DIR / "tensorboard")

script_name = ("run_qwen2.5_1.5b_MVaR_dapomath.sh" if DATASET == "dapo"
               else "run_qwen2.5_1.5b_MVaR_math.sh")
shutil.copy2(REPO / "scripts" / script_name, RUN_DIR / script_name)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
metadata = {
    "repo": REPO_URL, "commit": commit, "model": MODEL_ID,
    "method": METHOD, "dataset": DATASET, "run_name": RUN_NAME,
    "gsm8k_only": DATASET == "gsm8k",
    "gsm8k_preprocess_sha256": (hashlib.sha256(GSM8K_PREPROCESS_SOURCE.encode()).hexdigest()
                                if DATASET == "gsm8k" else None),
    "run_profile": RUN_PROFILE,
    "durable_backup_root": str(DURABLE_BACKUP_ROOT) if DURABLE_CHECKPOINTS else None,
    "resume_start_step": RESUME_START_STEP,
    "session_steps": SESSION_STEPS,
    "enable_thinking": ENABLE_THINKING,
    "thinking_dataset_sha256": (hashlib.sha256(THINKING_DATASET_PATH.read_bytes()).hexdigest()
                                 if ENABLE_THINKING is not None else None),
    "source_script": script_name, "gpu": GPU_INFO,
    "cuda_visible_devices": ENV.get("CUDA_VISIBLE_DEVICES"),
    "gsm8k_scorer_version": "final_numeric_v2",
    "custom_reward_sha256": hashlib.sha256(GSM8K_REWARD_PATH.read_bytes()).hexdigest(),
    "data_protocol": "Original files and question text; no resplit/subsampling. Qwen3 chat-template thinking mode explicitly configured.",
    "intentional_differences": {
        "model_override": not USE_REFERENCE_MODEL,
        "omit_math_dataset": DATASET == "gsm8k",
        "chat_template_enable_thinking": ENABLE_THINKING,
        "hybrid_estimator_and_loss": False,
        "gpu_world_size": 4,
        "rollout_tp": 1,
        "training": "Four-GPU BF16 LoRA, same global batch/micro-batch, CPU offload.",
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "runtime": "Isolated Python; vLLM V0; unique run/output paths.",
    },
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
if DATASET == "gsm8k":
    (RUN_DIR / "gsm8k_preprocess.py").write_text(GSM8K_PREPROCESS_SOURCE, encoding="utf-8")
freeze = subprocess.check_output([str(UV), "pip", "freeze", "--python", str(PY)], text=True)
(RUN_DIR / "requirements-freeze.txt").write_text(freeze, encoding="utf-8")
print(json.dumps(CONFIG, indent=2))
print("Run:", RUN_DIR)
print("Local checkpoints:", CKPT_DIR)
print("Số câu hỏi tối đa được lấy trong run:",
      CONFIG["trainer"]["total_training_steps"] * CONFIG["data"]["train_batch_size"],
      "(không phải số câu hỏi duy nhất hoặc cam kết đã đi qua toàn bộ tập)")

## 8. Kiểm tra dữ liệu, reward và ghi manifest

Giữ nguyên parquet train/test và prompt. Ghi số hàng/checksum, số mẫu sau lọc và thống kê nguồn.

Custom reward đã sửa sự không khớp prompt boxed/scorer #### của GSM8K. Cell này kiểm tra đáp án chuẩn ở cả hai định dạng với **đúng router cấu hình training**. Scorer chỉ nhận đáp án số rõ ràng, dùng dấu đáp án cuối, không nhặt một số bất kỳ từ lời giải.

**Với Qwen3:** mục này dùng đúng custom dataset của trainer, kiểm tra prompt token IDs thực sự có hậu tố thinking đã chọn. Khi tắt, `<think>...</think>` rỗng nằm trong prompt, không phải lời suy luận do model sinh. Scorer không đổi; bật thinking mà bị cắt trước đáp án cuối vẫn có thể nhận 0.


In [ ]:
audit = {"config_dir": str(REPO / "verl/trainer/config"),
         "config_name": CONFIG_NAME, "run_dir": str(RUN_DIR), "dataset": DATASET}
run_py("AUDIT = " + repr(audit) + "\n" + r'''
from pathlib import Path
import hashlib
import json
import importlib.util
import pandas as pd
from hydra import initialize_config_dir, compose
from transformers import AutoTokenizer
from verl.trainer.main_ppo import create_rl_dataset

with initialize_config_dir(config_dir=AUDIT["config_dir"], version_base=None):
    cfg = compose(config_name=AUDIT["config_name"])
spec = importlib.util.spec_from_file_location("riskpo_audit_reward", cfg.custom_reward_function.path)
reward_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reward_module)
score_fn = getattr(reward_module, cfg.custom_reward_function.name)
assert reward_module.GSM8K_SCORER_VERSION == "final_numeric_v2"
tokenizer = AutoTokenizer.from_pretrained(cfg.actor_rollout_ref.model.path)
manifest = {"files": [], "usable_rows": {}}
for role, paths in [("train", cfg.data.train_files), ("evaluation", cfg.data.val_files)]:
    for filename in paths:
        path = Path(filename)
        frame = pd.read_parquet(path)
        if AUDIT["dataset"] == "gsm8k":
            expected_split = 'train' if role == 'train' else 'test'
            assert len(paths) == 1 and path.name == expected_split + '.parquet'
            assert set(frame['data_source']) == {'openai/gsm8k'}, 'Unexpected dataset in GSM8K-only run'
            assert all(info['split'] == expected_split for info in frame['extra_info']), 'Wrong GSM8K split'
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        item = {"role": role, "path": str(path), "rows": len(frame),
                "sha256": digest.hexdigest(),
                "sources": {str(k): int(v) for k, v in frame["data_source"].value_counts().items()}}
        manifest["files"].append(item)
        print(item)
        gsm = frame[frame["data_source"] == "openai/gsm8k"]
        if len(gsm):
            truth = str(gsm.iloc[0]["reward_model"]["ground_truth"])
            for response in [r"\boxed{" + truth + "}", "#### " + truth]:
                assert score_fn("openai/gsm8k", response, truth) == 1, (response, truth)
            print("GSM8K boxed/#### checks passed:", reward_module.GSM8K_SCORER_VERSION)
    dataset = create_rl_dataset(list(paths), cfg.data, tokenizer, None, is_train=(role == "train"))
    manifest["usable_rows"][role] = len(dataset)
    assert len(dataset) > 0, f"{role} empty after filtering"
    if cfg.data.get("enable_thinking") is not None:
        enabled = cfg.data.enable_thinking
        row = dataset[0]
        messages = dataset.dataframe[0][cfg.data.prompt_key]
        expected_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=enabled)
        expected_ids = tokenizer.encode(expected_text, add_special_tokens=False)
        assert list(row["raw_prompt_ids"]) == expected_ids, 'Rollout prompt does not match thinking mode'
        manifest.setdefault("thinking_checks", {})[role] = {
            "enable_thinking": enabled, "raw_prompt_ids_match": True}
        print(role, "raw_prompt_ids check passed; enable_thinking =", enabled)
assert manifest["usable_rows"]["train"] >= cfg.data.train_batch_size
(Path(AUDIT["run_dir"]) / "data_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("After prompt filtering:", manifest["usable_rows"])
''')

## 9. Resolve Hydra và xác minh RiskPO/4 GPU

Kiểm tra estimator RiskPO, vanilla loss, quantile tracking, LoRA/BF16,
FSDP=4, TP/SP=1, global batch chia hết, save/load đủ bốn rank và resume world size.
Không bật validation. Lưu resolved config để đối chiếu với notebook một GPU.

In [ ]:
COMMAND = [str(PY), "-m", "verl.trainer.main_ppo", "--config-name", CONFIG_NAME]
resolved = subprocess.check_output(
    COMMAND + ["--cfg", "job", "--resolve"], cwd=REPO, env=ENV, text=True
)
(RUN_DIR / "resolved_config.yaml").write_text(resolved, encoding="utf-8")
print(resolved)
run_py("AUDIT = " + repr(audit) + "\n" + "METHOD = " + repr(METHOD) + "\n" + r'''
from hydra import initialize_config_dir, compose
import torch

with initialize_config_dir(config_dir=AUDIT["config_dir"], version_base=None):
    cfg = compose(config_name=AUDIT["config_name"])
assert torch.cuda.device_count() == cfg.trainer.n_gpus_per_node == 4
assert cfg.custom_reward_function.name == "compute_dataset_score"
from pathlib import Path
assert Path(cfg.custom_reward_function.path).is_file()
if cfg.data.get("enable_thinking") is not None:
    assert cfg.data.custom_cls.name == "ThinkingModeDataset"
    assert Path(cfg.data.custom_cls.path).is_file()
assert cfg.actor_rollout_ref.model.lora_rank > 0
assert cfg.actor_rollout_ref.actor.fsdp_config.model_dtype == "bf16"
assert cfg.trainer.n_gpus_per_node == 4
assert cfg.trainer.nnodes == 1
assert cfg.actor_rollout_ref.actor.fsdp_config.fsdp_size == 4
assert cfg.actor_rollout_ref.actor.loss_agg_mode == "token-mean"
assert cfg.data.train_batch_size * cfg.actor_rollout_ref.rollout.n % 4 == 0
assert cfg.actor_rollout_ref.actor.ppo_mini_batch_size * cfg.actor_rollout_ref.rollout.n % 4 == 0
assert cfg.actor_rollout_ref.rollout.tensor_model_parallel_size == 1
assert cfg.actor_rollout_ref.actor.ulysses_sequence_parallel_size == 1
assert cfg.actor_rollout_ref.rollout.max_num_seqs == 24
assert cfg.actor_rollout_ref.rollout.engine_kwargs.vllm.max_num_seqs == 24
assert cfg.actor_rollout_ref.rollout.load_format == "safetensors"
assert cfg.actor_rollout_ref.rollout.layered_summon
assert cfg.data.train_batch_size % cfg.actor_rollout_ref.actor.ppo_mini_batch_size == 0
assert cfg.trainer.test_freq == -1
if cfg.trainer.get("durable_checkpoint"):
    assert cfg.trainer.save_freq == 5
    assert cfg.trainer.max_actor_ckpt_to_keep == 2
    assert not cfg.trainer.del_local_ckpt_after_load
    assert cfg.trainer.durable_checkpoint.session_steps > 0
    assert set(cfg.actor_rollout_ref.actor.checkpoint.save_contents) == {"model", "optimizer", "extra"}
    assert set(cfg.actor_rollout_ref.actor.checkpoint.load_contents) == {"model", "optimizer", "extra"}
    if cfg.trainer.resume_mode == "resume_path":
        assert (Path(cfg.trainer.resume_from_path) / "complete.json").is_file()
else:
    assert cfg.trainer.save_freq == (5 if cfg.trainer.total_training_steps == 5 else 10)
assert not cfg.trainer.val_before_train
assert cfg.actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu == 1
assert not cfg.actor_rollout_ref.rollout.val_kwargs.do_sample
assert cfg.actor_rollout_ref.rollout.val_kwargs.n == 1
assert METHOD == "riskpo"
assert cfg.algorithm.adv_estimator == "grpo_bundle_RVaR_quantile_tracking"
assert cfg.algorithm.quantile_tracking
assert cfg.actor_rollout_ref.actor.policy_loss.loss_mode == "vanilla"
print("Resolved config checks passed.")
''')

## 10. Training RiskPO LoRA và export adapter

Chạy cell này mới bắt đầu training. Một lệnh Python khởi tạo Ray và cả bốn worker;
không chạy cell bốn lần và không bọc main_ppo bằng torchrun.
GSM8K vẫn 100 response/global step, 25/rank, microbatch 1.

Giữ cách chấm final_numeric_v2. Theo dõi loss/gradient hữu hạn, reward/response và
log quantile tracking. Nếu chọn smoke: cần hoàn tất 5 bước, lưu đủ bốn shard/rank,
backup checksum hợp lệ và export adapter đọc được. Không suy ra chất lượng từ smoke.

Save/export có thể dùng nhiều RAM dù LoRA. Cell finally không được bảo đảm chạy
khi máy bị thu hồi; backup mỗi 5 bước là điểm phục hồi. Không tự đổi batch hoặc token limit khi OOM.

In [ ]:
def required_local_checkpoint_files():
    names = ["data.pt", "actor/fsdp_config.json", "actor/lora_adapter/adapter_config.json",
             "actor/lora_adapter/adapter_model.safetensors"]
    names += [f"actor/{kind}_world_size_4_rank_{rank}.pt"
              for rank in range(4) for kind in ("model", "optim", "extra_state")]
    if DURABLE_CHECKPOINTS:
        names.append("trainer_state.pt")
    return names

def find_latest_adapter(root):
    if not root.is_dir():
        return None
    candidates = sorted(
        (p for p in root.glob("global_step_*") if p.name.rsplit("_", 1)[-1].isdigit()),
        key=lambda p: int(p.name.rsplit("_", 1)[-1]), reverse=True,
    )
    for step in candidates:
        adapter = step / "actor/lora_adapter"
        required = [step / name for name in required_local_checkpoint_files()]
        if all(p.is_file() and p.stat().st_size > 0 for p in required):
            try:
                fsdp = json.loads((step / "actor/fsdp_config.json").read_text())
                if fsdp.get("world_size") != 4:
                    continue
                if DURABLE_CHECKPOINTS:
                    durable.inventory(step)
            except (ValueError, OSError, KeyError):
                continue
            return adapter
    return None

def export_latest_adapter():
    adapter = find_latest_adapter(CKPT_DIR)
    if adapter is None:
        print("Chưa thấy checkpoint hoàn chỉnh có adapter; xem log save.")
        return None
    # Validate inside the isolated environment; do not require safetensors in the notebook kernel.
    run_py("ADAPTER = " + repr(str(adapter)) + "\n" + r'''
from pathlib import Path
import json
from safetensors import safe_open
p = Path(ADAPTER)
config = json.loads((p / "adapter_config.json").read_text())
assert config["peft_type"] == "LORA" and config["r"] > 0
with safe_open(str(p / "adapter_model.safetensors"), framework="pt", device="cpu") as f:
    assert any("lora_" in key for key in f.keys()), "No LoRA tensors in saved adapter"
''')
    dest = RUN_DIR / "adapters" / adapter.parent.parent.name
    if dest.exists():
        print("Đích đã tồn tại, không ghi đè:", dest)
        return dest
    shutil.copytree(adapter, dest)
    print("Adapter đã export:", dest, "| Base:", MODEL_ID)
    return dest

ACKNOWLEDGE_REFERENCE_LIMITATIONS = True
if not ACKNOWLEDGE_REFERENCE_LIMITATIONS:
    raise RuntimeError(
        "Hãy đọc giới hạn phần đầu và cảnh báo reward ở bước 8, "
        "sau đó đặt ACKNOWLEDGE_REFERENCE_LIMITATIONS=True nếu muốn chạy."
    )
check_gpu_count()
(RUN_DIR / "command.json").write_text(json.dumps(COMMAND, indent=2), encoding="utf-8")
return_code = None
ADAPTER_DIR = None
try:
    with (RUN_DIR / "train.log").open("w", encoding="utf-8", buffering=1) as log:
        process = subprocess.Popen(
            COMMAND, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True,
        )
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            return_code = process.wait()
        except KeyboardInterrupt:
            import signal
            os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGTERM)
            raise
finally:
    try:
        ADAPTER_DIR = export_latest_adapter()
    except Exception as export_error:
        print("Export adapter thất bại:", export_error)

if return_code != 0:
    raise RuntimeError(f"Training lỗi: exit={return_code}; xem {RUN_DIR / 'train.log'}")
if ADAPTER_DIR is None:
    raise RuntimeError("Training thoát nhưng chưa xác minh được adapter; chạy lại bước 11.")
if DURABLE_CHECKPOINTS:
    completed_step = int(ADAPTER_DIR.name.rsplit("_", 1)[-1])
    if completed_step < CONFIG["trainer"]["total_training_steps"]:
        print("SESSION PAUSED at", completed_step, "- training target is not complete.")
    else:
        print("TOTAL TRAINING TARGET COMPLETE at", completed_step)
    print("Verified backups:", DURABLE_BACKUP_ROOT)
    print("Next session: keep the same settings and set RESUME_BACKUP_ROOT =", repr(str(DURABLE_BACKUP_ROOT)))
print("Session ended. Adapter:", ADAPTER_DIR)

## 11. Export lại adapter và backup thủ công tùy chọn

Chỉ chọn checkpoint local đủ model/optimizer/extra của cả bốn rank, dataloader
và adapter. Khi durable bật còn yêu cầu trainer_state. Adapter được kiểm tra
bằng safetensors trong Python riêng trước khi export. Không cần merge FSDP shards
để dùng adapter cùng base Qwen3-4B.

Durable đã backup mỗi 5 bước; COPY_FULL_CHECKPOINT chỉ tạo thêm bản sao thủ công.
Không dùng adapter export để thay cho full checkpoint khi resume.

In [ ]:
ADAPTER_DIR = export_latest_adapter()
if ADAPTER_DIR is None:
    raise RuntimeError("Chưa có adapter đã xác minh. Kiểm tra train.log.")
print("Adapter:", ADAPTER_DIR)

COPY_FULL_CHECKPOINT = False  # Extra manual copy only; durable backups already saved every 5 steps
if COPY_FULL_CHECKPOINT:
    latest_adapter = find_latest_adapter(CKPT_DIR)
    if latest_adapter is None:
        raise RuntimeError("Không tìm thấy checkpoint tương ứng.")
    step = latest_adapter.parent.parent
    required = [step / name for name in required_local_checkpoint_files()]
    assert all(p.is_file() and p.stat().st_size > 0 for p in required)
    target = RUN_DIR / "checkpoints" / step.name
    if target.exists():
        raise FileExistsError(f"Không ghi đè backup: {target}")
    size_gib = sum(p.stat().st_size for p in step.rglob("*") if p.is_file()) / 2**30
    print(f"Sao lưu {size_gib:.2f} GiB đến {target}")
    shutil.copytree(step, target)
    print("Đã sao lưu full checkpoint.")
else:
    print("Adapter đã export. Durable backup:" if DURABLE_CHECKPOINTS else "Checkpoint chỉ ở local:", DURABLE_BACKUP_ROOT if DURABLE_CHECKPOINTS else CKPT_DIR)

## 12. ZIP adapter để tải về (tùy chọn)

Bật `DOWNLOAD_ADAPTER=True` để tạo ZIP và tải về nếu đang ở Colab. Adapter không chứa trọng số base. Khi inference, dùng PEFT nạp base theo `MODEL_ID` rồi gắn adapter bằng `PeftModel.from_pretrained`. Với Qwen3, truyền `enable_thinking=False` vào `tokenizer.apply_chat_template` để đánh giá cùng chế độ run mặc định này; adapter không tự lưu cờ thinking.

Log, resolved config, manifest và thư viện nằm trong `RUN_DIR`. TeleMath cần pipeline benchmark riêng; không được train trong notebook này.


In [ ]:
DOWNLOAD_ADAPTER = False
if DOWNLOAD_ADAPTER:
    if ADAPTER_DIR is None or not ADAPTER_DIR.is_dir():
        raise RuntimeError("Hãy export adapter ở bước 11 trước.")
    zip_base = WORK_ROOT / (RUN_NAME + "_" + ADAPTER_DIR.name + "_adapter")
    if zip_base.with_suffix(".zip").exists():
        raise FileExistsError("ZIP đã tồn tại, không ghi đè.")
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=ADAPTER_DIR)
    print("ZIP:", zip_path)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(zip_path)
else:
    print("Bỏ qua tải ZIP; adapter đã export vẫn nằm trong RUN_DIR.")


## Phạm vi so sánh và trạng thái kiểm chứng

- Baseline RiskPO gốc dùng quantile tracking + vanilla, cùng GSM8K, Qwen3-4B,
  scorer, LoRA, thinking và ngân sách global batch như thiết kế nguồn.
- Chuyển từ 1 sang 4 GPU không tăng số câu hỏi/rollout theo step. 200 bước GSM8K
  tương ứng 4.000 lượt câu hỏi/20.000 response, không phải đã duyệt hết dataset.
- Không dùng Telemath để train hoặc tuning. GSM8K test được giữ cho đánh giá riêng,
  validation training vẫn tắt. Reward training không phải test accuracy.
- Khi so baseline và hybrid, nên chạy cả hai trên cùng phần cứng/topology nếu
  muốn kiểm soát thêm ảnh hưởng batching/số học và so sánh tốc độ.
- RiskPO thuật toán được giữ; LoRA/GSM8K-only/scorer sửa/batch nhỏ khác thiết lập
  bài báo. Notebook này không phải tái lập kết quả full-parameter RiskPO.
- Notebook mới xóa toàn bộ output/execution count của bản nguồn. Kiểm tra local
  chỉ xác minh cấu trúc, cấu hình và giao thức backup bằng file giả lập.
  Chưa chạy end-to-end trên 4 A100; NCCL, FSDP/LoRA, VRAM, save/load GPU và tốc độ
  phải được xác nhận trên máy thật. Các cell preflight hỗ trợ phát hiện lỗi sớm.